# **Séance 4 : Programmation orientée objet 2 : héritage et décorateurs**

En séance 3, nous avons créé la classe `Manuscrit` pour décrire les bestiaires de notre corpus. Dans ce notebook, on apprend à :

- créer des **classes enfants** qui héritent de `Manuscrit` (**héritage**) ;
- **indiquer** et **vérifier** le type des données, et **gérer les erreurs** (typage et exceptions) ;
- ajouter un comportement à une fonction sans modifier son code (**décorateurs**).

---

## **0. Préparation**

Le fichier `bestiaire.py`, créé en séance 2, doit se trouver dans le même dossier que ce notebook.

In [ ]:
import bestiaire

titre = "III, 14. De la colombe."
creatures = ["dragon", "licorne", "lion", "panthère", "phénix"]

Voici la classe `Manuscrit` telle que nous l'avons construite en séance 3. Exécutez la cellule : elle servira de **classe parente** dans toute la partie 1.

In [ ]:
class Manuscrit:
    """Un manuscrit du corpus de bestiaires."""

    genre = "bestiaire"

    def __init__(self, cote, bibliotheque, date):
        self.cote = cote
        self.bibliotheque = bibliotheque
        self.date = date
        self.creatures = []

    def cote_complete(self):
        """Renvoyer la bibliothèque et la cote."""
        return f"{self.bibliotheque}, {self.cote}"

    def siecle(self):
        """Renvoyer le siècle du manuscrit."""
        return bestiaire.siecle(self.date)

    def ajouter_creature(self, nom):
        """Ajouter une créature à la liste du manuscrit."""
        self.creatures.append(nom)

    def fiche(self):
        """Renvoyer la fiche du manuscrit."""
        return (f"{self.cote_complete()} ({self.genre})\n"
                f"  Siècle : {self.siecle()}\n"
                f"  Créatures : {self.creatures}")


manuscrit = Manuscrit("MS 24", "Aberdeen, University Library", 1200)
manuscrit.ajouter_creature("licorne")
print(manuscrit.fiche())

---

## **1. L'héritage**

L'**héritage** permet à une classe, appelée **classe enfant** ou **sous-classe**, d'hériter des attributs et des méthodes d'une autre classe, appelée **classe parente** ou **super-classe**. Il modélise une relation « **est un** » : un manuscrit numérisé **est un** manuscrit.

### **1.1 Créer une classe enfant**

On indique la classe parente entre parenthèses.

In [ ]:
class ManuscritNumerise(Manuscrit):
    pass


manuscrit_oxford = ManuscritNumerise("MS. Ashmole 1511", "Oxford, Bodleian Library", 1210)

print(manuscrit_oxford.cote_complete())
print(manuscrit_oxford.siecle())
print(manuscrit_oxford.genre)

Sans une ligne de code de plus, `ManuscritNumerise` possède tous les attributs et toutes les méthodes de `Manuscrit`.

### **1.2 Ajouter un attribut : `super().__init__()`**

Un manuscrit numérisé a un attribut de plus : l'adresse de son **manifeste IIIF**, le fichier qui décrit ses images numérisées (nous y reviendrons en séance 7).

`super()` permet d'appeler les méthodes de la classe parente. `super().__init__()` appelle le `__init__` de `Manuscrit`, qui crée les attributs hérités :

- on **n'écrit pas `self`** dans les arguments : Python le passe automatiquement ;
- **bonne pratique** : on l'appelle **en premier**, avant d'initialiser les attributs propres à la classe enfant.

In [ ]:
class ManuscritNumerise(Manuscrit):
    """Un manuscrit du corpus dont les images sont accessibles en IIIF."""

    def __init__(self, cote, bibliotheque, date, manifeste):
        super().__init__(cote, bibliotheque, date)   # attributs hérités
        self.manifeste = manifeste                   # attribut propre


# Ajout du manifeste IIIF
manuscrit_oxford = ManuscritNumerise("MS. Ashmole 1511", "Oxford, Bodleian Library", 1210,
                                     "https://iiif.bodleian.ox.ac.uk/iiif/manifest/faeff7fb-f8a7-44b5-95ed-cff9a9ffd198.json")

print(manuscrit_oxford.cote_complete())
print(manuscrit_oxford.manifeste)
print(manuscrit_oxford.creatures)

`cote`, `bibliotheque` et `date` sont transmis à la classe parente, qui crée aussi la liste vide `creatures`. `manifeste` est propre à la classe enfant.

#### **Exercice 1 : Le `super()` oublié**

1. Exécutez la cellule ci-dessous : l'objet est créé sans erreur.
2. Décommentez la dernière ligne et exécutez à nouveau. Quelle erreur obtenez-vous ? Pourquoi ?
3. Corrigez la classe dans la cellule de solution.

In [ ]:
class ManuscritNumerise(Manuscrit):
    """Un manuscrit du corpus dont les images sont accessibles en IIIF."""

    def __init__(self, cote, bibliotheque, date, manifeste):
        self.manifeste = manifeste


manuscrit = ManuscritNumerise("Cod. 318", "Bern, Burgerbibliothek", 830, 
                              "https://www.e-codices.unifr.ch/metadata/iiif/bbb-0318/manifest.json")
# print(manuscrit.cote_complete())

*Solution :*

In [ ]:
# Votre code ici

### **1.3 Redéfinir une méthode avec `super()`**

Une classe enfant peut écrire sa propre version d'une méthode héritée : pour ses objets, Python utilise alors cette nouvelle version. Avec `super().methode()`, elle peut **réutiliser** la version de la classe parente et l'**étendre**.

La fiche d'un manuscrit numérisé reprend celle de `Manuscrit` et ajoute une ligne pour le manifeste.

In [ ]:
class ManuscritNumerise(Manuscrit):
    """Un manuscrit du corpus dont les images sont accessibles en IIIF."""

    def __init__(self, cote, bibliotheque, date, manifeste):
        super().__init__(cote, bibliotheque, date)
        self.manifeste = manifeste

    def fiche(self):
        """Renvoyer la fiche du manuscrit, avec son manifeste."""
        return f"{super().fiche()}\n  Manifeste : {self.manifeste}"


manuscrit_oxford = ManuscritNumerise("MS. Ashmole 1511", "Oxford, Bodleian Library", 1210,
                                     "https://iiif.bodleian.ox.ac.uk/iiif/manifest/faeff7fb-f8a7-44b5-95ed-cff9a9ffd198.json")
print(manuscrit_oxford.fiche())

`super().fiche()` **renvoie** la fiche de `Manuscrit`, sur trois lignes, et la classe enfant y ajoute une quatrième ligne. Les objets de la classe parente ne sont pas concernés :

In [ ]:
manuscrit = Manuscrit("MS 24", "Aberdeen, University Library", 1200)
print(manuscrit.fiche())

#### **Exercice 2 : La fiche incomplète**

Exécutez la cellule ci-dessous : la fiche n'affiche que le manifeste. Pourquoi ? Corrigez la méthode `fiche()` dans la cellule de solution.

In [ ]:
class ManuscritNumerise(Manuscrit):
    """Un manuscrit du corpus dont les images sont accessibles en IIIF."""

    def __init__(self, cote, bibliotheque, date, manifeste):
        super().__init__(cote, bibliotheque, date)
        self.manifeste = manifeste

    def fiche(self):
        """Renvoyer la fiche du manuscrit, avec son manifeste."""
        super().fiche()
        return f"  Manifeste : {self.manifeste}"


manuscrit_oxford = ManuscritNumerise("MS. Ashmole 1511", "Oxford, Bodleian Library", 1210,
                                     "https://iiif.bodleian.ox.ac.uk/iiif/manifest/faeff7fb-f8a7-44b5-95ed-cff9a9ffd198.json")
print(manuscrit_oxford.fiche())

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 3 : Classer les créatures du bestiaire**

Les bestiaires médiévaux sont organisés en livres : les bêtes, les oiseaux, les serpents…

1. Créez une classe `Creature` avec :
    - un attribut de classe `livre`, qui vaut `"créatures"` ;
    - un attribut d'instance `nom` ;
    - une méthode `decrire()` qui **renvoie** une phrase de la forme `"lion : livre des bêtes"`.
2. Créez trois classes enfants, `Bete`, `Oiseau` et `Serpent`, qui ne font que **redéfinir** l'attribut `livre` (`"bêtes"`, `"oiseaux"`, `"serpents"`).
3. Créez une liste `creatures_classees` qui contient les cinq créatures de la liste `creatures`, chacune dans la bonne classe : le dragon est un serpent, le phénix un oiseau, les trois autres sont des bêtes.
4. Affichez la description de chaque créature avec une boucle.

*Indice : dans `decrire()`, `self.livre` cherche d'abord l'attribut dans la classe de l'objet, puis dans sa classe parente.*

*Solution :*

In [ ]:
# Votre code ici

---

## **2. Typage et exceptions**

### **2.1 Les annotations de type**

Python est un langage **dynamiquement typé** : il ne vérifie pas le type des arguments passés à une fonction. Les **annotations de type** permettent d'**indiquer** le type attendu :

- `date: int` indique que le paramètre **devrait** être un entier ;
- `-> str` indique que la fonction **devrait** renvoyer une chaîne de caractères.

Elles servent à la documentation et aux outils de vérification, comme ceux de VS Code.

In [ ]:
def siecle(date: int) -> str:
    """Renvoyer le siècle d'une date."""
    if date <= 1200:
        resultat = "XIIe siècle"
    else:
        resultat = "XIIIe siècle"
    return resultat


print(siecle(1210))

Les annotations sont **non contraignantes** : Python accepte l'appel avec une chaîne. Décommentez la ligne pour voir l'erreur.

In [ ]:
# siecle("vers 1210")
# TypeError: '<=' not supported between instances of 'str' and 'int'
# L'erreur ne survient qu'à la comparaison, et le message ne dit rien de l'annotation.

#### **Exercice 4 : Annoter une fonction**

1. Écrivez une fonction `ecart(date1, date2)` qui renvoie le nombre d'années entre deux dates, avec les annotations de type de ses paramètres et de sa valeur de retour.
2. Testez-la avec les dates des manuscrits d'Aberdeen et d'Oxford.
3. Que renvoie `ecart(1200.0, 1210)` ? Pourquoi Python ne signale-t-il rien ?

*Solution :*

In [ ]:
# Votre code ici

### **2.2 Vérifier un type et lever une exception : `isinstance()` et `raise`**

Une **exception** est une erreur qui survient pendant l'exécution et l'interrompt : `TypeError`, `NameError`, `AttributeError`…

- `isinstance(valeur, type)` renvoie `True` si la valeur est du type indiqué, `False` sinon ;
- `raise NomDeLErreur("Message d'erreur")` déclenche **volontairement** une exception, avec un message clair.

Pour des projets plus complexes, des bibliothèques comme **Pydantic** vérifient les types automatiquement.

In [ ]:
print(isinstance(1210, int))
print(isinstance("vers 1210", int))

In [ ]:
def siecle(date: int) -> str:
    """Renvoyer le siècle d'une date."""
    if not isinstance(date, int):
        raise TypeError("la date doit être un nombre entier")
    if date <= 1200:
        resultat = "XIIe siècle"
    else:
        resultat = "XIIIe siècle"
    return resultat


print(siecle(1210))
# siecle("vers 1210")
# TypeError: la date doit être un nombre entier

Décommentez l'appel : l'erreur est levée **dès l'entrée** dans la fonction, avec un message qui dit ce qui ne va pas.

### **2.3 Gérer les exceptions : `try` et `except`**

Une exception non gérée **arrête** le programme. La gérer permet de **décider quoi faire** en cas d'erreur, et de continuer :

- le bloc `try` contient le code qui pourrait provoquer une exception ;
- le bloc `except` contient le code à exécuter si cette exception survient.

Une date saisie en texte ne doit pas arrêter le traitement des autres :

In [ ]:
for date in [1200, "vers 1210"]:
    try:
        print(siecle(date))
    except TypeError:
        print(f"Date à vérifier : {date}")

La boucle ne s'arrête pas : la date fautive est signalée, et la suivante est traitée.

### **2.4 Créer ses propres exceptions**

Une exception personnalisée est une classe qui **hérite** de `Exception` : une `DateInvalide` **est une** exception, et elle hérite de tout son fonctionnement. On la lève et on la gère comme une exception native.

In [ ]:
class DateInvalide(Exception):
    pass


def siecle(date: int) -> str:
    """Renvoyer le siècle d'une date."""
    if not isinstance(date, int):
        raise DateInvalide("la date doit être un nombre entier")
    if date <= 1200:
        resultat = "XIIe siècle"
    else:
        resultat = "XIIIe siècle"
    return resultat


for date in [1200, "vers 1210"]:
    try:
        print(siecle(date))
    except DateInvalide:
        print(f"Date à vérifier : {date}")

#### **Exercice 5 : Compter les manuscrits par siècle**

Voici des dates saisies dans un catalogue (fictif). Certaines ne sont pas des nombres entiers.

```python
dates = [1200, "vers 1210", 1250, "XIIIe siècle", 1180]
```

Avec une boucle, la fonction `siecle()` de la partie 2.4, `try` et `except`, construisez :

- un dictionnaire `compte` qui associe à chaque siècle le nombre de dates correspondantes ;
- une liste `a_verifier` qui contient les dates refusées.

Résultat attendu : `{'XIIe siècle': 2, 'XIIIe siècle': 1}` et `['vers 1210', 'XIIIe siècle']`.

*Solution :*

In [ ]:
dates = [1200, "vers 1210", 1250, "XIIIe siècle", 1180]

# Votre code ici

---

## **3. Les décorateurs**

Un **décorateur** est une fonction qui prend en entrée une autre fonction, et renvoie une nouvelle fonction au comportement modifié ou étendu. Il permet de modifier le comportement d'une fonction **sans changer son code source**.

### **3.1 Construire un décorateur pas à pas**

##### **Étape 1 : une fonction simple**

In [ ]:
def afficher_titre():
    print(titre)


afficher_titre()

##### **Étape 2 : une fonction qui prend une fonction en paramètre**

En Python, une fonction est un objet comme un autre : on peut la passer en argument, la renvoyer, la stocker dans une variable.

`annoncer` reçoit une fonction, définit à l'intérieur une **nouvelle fonction** qui exécute du code avant et après elle, puis **renvoie** cette nouvelle fonction.

In [ ]:
def annoncer(fonction):
    def nouvelle_fonction():
        print("Avant l'exécution de la fonction")
        resultat = fonction()
        print("Après l'exécution de la fonction")
        return resultat
    return nouvelle_fonction

##### **Étape 3 : décorer la fonction**

In [ ]:
afficher_titre = annoncer(afficher_titre)

afficher_titre()

Le nom `afficher_titre` désigne maintenant `nouvelle_fonction`, qui appelle l'ancienne `afficher_titre` entre ses deux messages.

##### **Étape 4 : la syntaxe avec `@`**

Écrire `@annoncer` juste au-dessus de la définition revient exactement au même que `afficher_titre = annoncer(afficher_titre)` :

In [ ]:
@annoncer
def afficher_titre():
    print(titre)


afficher_titre()

**Ce qui se passe :**

- `annoncer` est appelé avec `afficher_titre` comme argument ;
- il définit `nouvelle_fonction`, qui ajoute du code avant et après `afficher_titre` ;
- il renvoie `nouvelle_fonction`, qui remplace `afficher_titre` ;
- quand on appelle `afficher_titre()`, c'est en réalité `nouvelle_fonction()` qui s'exécute.

### **3.2 Décorer une fonction qui a des paramètres : `*args` et `**kwargs`**

`nouvelle_fonction()` n'accepte aucun argument : `annoncer` ne peut donc pas décorer `siecle()`. Décommentez les lignes pour voir l'erreur.

In [ ]:
# siecle_annoncee = annoncer(siecle)
# siecle_annoncee(1210)
# TypeError: annoncer.<locals>.nouvelle_fonction() takes 0 positional arguments but 1 was given

**Rappel de la séance 2** : `*args` regroupe les arguments positionnels dans un **tuple**, et `**kwargs` les arguments nommés dans un **dictionnaire**.

Dans un décorateur :

- dans le `def`, les étoiles **regroupent** tous les arguments reçus, quels qu'ils soient ;
- à l'appel de `fonction`, elles les **déballent** pour les transmettre tels quels.

In [ ]:
def annoncer(fonction):
    def nouvelle_fonction(*args, **kwargs):
        print("Avant l'exécution de la fonction")
        resultat = fonction(*args, **kwargs)
        print("Après l'exécution de la fonction")
        return resultat
    return nouvelle_fonction


@annoncer
def siecle(date: int) -> str:
    """Renvoyer le siècle d'une date."""
    if not isinstance(date, int):
        raise DateInvalide("la date doit être un nombre entier")
    if date <= 1200:
        resultat = "XIIe siècle"
    else:
        resultat = "XIIIe siècle"
    return resultat


print(siecle(1210))

« XIIIe siècle » s'affiche **en dernier** : `siecle()` **renvoie** le siècle, et c'est le `print()` extérieur qui l'affiche, une fois le décorateur terminé.

Le même décorateur fonctionne aussi avec une fonction sans paramètre :

In [ ]:
@annoncer
def afficher_titre():
    print(titre)


afficher_titre()

#### **Exercice 6 : Un décorateur qui signale le calcul**

1. Créez un décorateur `signaler` qui affiche « Calcul en cours… » avant l'appel de la fonction et « Calcul terminé. » après.
2. Appliquez-le à la fonction `ecart()` de l'exercice 4 : recopiez sa définition et ajoutez seulement la ligne `@signaler` au-dessus. Testez-la.
3. Faites de même avec la fonction `siecle()` de la partie 2.4.

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 7 (facultatif) : Vérifier la date avec un décorateur**

1. Créez un décorateur `verifier_date` qui lève `DateInvalide` si le **premier argument** de la fonction n'est pas un entier, et qui appelle la fonction sinon.
2. Appliquez-le à une version de `siecle()` **sans** `isinstance()` : la vérification est faite par le décorateur.
3. Testez avec `1210`, puis avec `"vers 1210"` dans un `try` / `except`.

*Indice : dans `nouvelle_fonction(*args, **kwargs)`, le premier argument est `args[0]`.*

*Solution :*

In [ ]:
# Votre code ici

---

### **Conclusion**

Dans ce notebook, nous avons :

- créé une **classe enfant**, `ManuscritNumerise`, qui hérite de `Manuscrit`, lui ajoute un attribut avec `super().__init__()` et redéfinit sa fiche avec `super().fiche()` ;
- **indiqué** les types avec des annotations, puis **vérifié** une date avec `isinstance()` et `raise`, **géré** les erreurs avec `try` et `except`, et créé notre propre exception, `DateInvalide` ;
- construit des **décorateurs**, rendus compatibles avec toutes les fonctions grâce à `*args` et `**kwargs`.

À la prochaine séance, nous apprendrons à **lire et enregistrer** les données de notre corpus au format **JSON**.